# House Prices - Fase 3: Modelo Baseline

Nesta fase vamos treinar nossos primeiros modelos de Machine Learning (ML).

## O que e Machine Learning?

ML e um subconjunto da Inteligencia Artificial onde o computador **aprende padroes a partir dos dados** em vez de ser explicitamente programado.

**Analogia:**
- Programacao tradicional: voce escreve regras -> 'Se a casa tem 3 quartos E 2 banheiros, custa R$ 300k'
- ML: voce da milhares de exemplos -> o algoritmo DESCOBRE as regras sozinho

## O que vamos aprender nesta fase:

1. **Como funciona a Regressao Linear** (Ridge)
2. **Como funciona o Random Forest**
3. **Como avaliar modelos** (RMSE, RMSLE, Cross-Validation)
4. **Como gerar previsoes** para o Kaggle

## Fluxo completo:

```
Dados limpos (Fase 2)
       |
       v
   Treinar modelo
       |
       v
   Avaliar metricas
       |
       v
   Gerar previsoes
       |
       v
   Enviar ao Kaggle
```

## Passo 1: Carregar os dados

Vamos carregar os dados limpos que preparamos na Fase 2.

**Por que separamos X (features) e y (target)?**

- **X** = variaveis de entrada (features). Ex: area, quartos, qualidade
- **y** = variavel que queremos prever (target). Ex: preco da casa

O modelo aprende a relacao: `f(X) = y`

**Por que temos X_train e X_test separados?**

- **X_train**: dados para TREINAR o modelo (com o target y_train)
- **X_test**: dados para AVALIAR o modelo (sem o target - o Kaggle esconde!)

O modelo NUNCA ve o X_test durante o treino. Isso e como uma prova: o modelo estuda com um material (train) e e avaliado com questoes que nunca viu (test).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import os
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_val_score, KFold
from sklearn.metrics import mean_squared_error

sns.set_style('whitegrid')

# Detectar diretorio base automaticamente
# Funciona tanto no VS Code (cwd=projeto) quanto no nbconvert (cwd=notebooks/)
_cwd = os.getcwd()
if _cwd.endswith('notebooks'):
    BASE_DIR = os.path.dirname(_cwd)
else:
    BASE_DIR = _cwd

DATA_DIR = os.path.join(BASE_DIR, 'data')
REPORTS_DIR = os.path.join(BASE_DIR, 'reports')
MODELS_DIR = os.path.join(BASE_DIR, 'models')
os.makedirs(MODELS_DIR, exist_ok=True)

# Carregar dados limpos da Fase 2
X_train = pd.read_csv(os.path.join(DATA_DIR, 'X_train_clean.csv'))
y_train = pd.read_csv(os.path.join(DATA_DIR, 'y_train_clean.csv')).squeeze()
X_test = pd.read_csv(os.path.join(DATA_DIR, 'X_test_clean.csv'))
test_original = pd.read_csv(os.path.join(DATA_DIR, 'test.csv'))

print('Dados carregados com sucesso!')
print(f'  X_train: {X_train.shape[0]} casas, {X_train.shape[1]} features')
print(f'  y_train: {y_train.shape[0]} precos')
print(f'  X_test:  {X_test.shape[0]} casas (para prever)')
print(f'\n  Media do preco (log): {y_train.mean():.4f}')
print(f'  Desvio padrao (log): {y_train.std():.4f}')
print(f'\n  Exemplo de X_train (primeira linha):')
print(X_train.iloc[0].head(10))

## Passo 2: Entender as metricas de avaliacao

Como sabemos se um modelo e bom ou ruim? Precisamos de metricas!

### RMSE (Root Mean Squared Error)

$$RMSE = \sqrt{\frac{1}{n} \sum_{i=1}^{n} (y_i - \hat{y}_i)^2}$$

**Traducao:**
1. Calcule a diferenca entre previsao e valor real: `(y_i - y_hat_i)`
2. Eleve ao quadrado (para ficar positivo e penalizar erros grandes)
3. Calcule a media
4. Tire a raiz quadrada

**Analogia:**
Imagine que voce atira flechas em um alvo. RMSE e a distancia media do centro.
- Erros pequenos: quase no centro
- Erros grandes: longe do centro (e ao quadrado, penaliza mais!)

**RMSE menor = melhor**

### RMSLE (Root Mean Squared Log Error)

$$RMSLE = \sqrt{\frac{1}{n} \sum_{i=1}^{n} (\log(y_i + 1) - \log(\hat{y}_i + 1))^2}$$

**Diferenca do RMSE:** aplica log ANTES de calcular o erro.

**Por que usar log?**
- Penaliza mais quem SUBESTIMA do que quem SUPERESTIMA
- Ex: prever R$ 100k quando e R$ 200k e PIOR que prever R$ 300k quando e R$ 200k
- Isso faz sentido no mercado imobiliario: subestimar e mais arriscado

**RMSLE e a metrica oficial desta competicao do Kaggle!**

In [ ]:
def rmsle(y_true, y_pred):
    """Calcula RMSLE (Root Mean Squared Log Error)
    
    y_true: valores reais (escala log)
    y_pred: valores previstos (escala log)
    """
    # Garantir que previsoes nao sejam negativas
    y_pred = np.maximum(y_pred, 0)
    return np.sqrt(mean_squared_error(np.log1p(y_true), np.log1p(y_pred)))


def evaluate_rmse(model, X, y, cv=5):
    """Avalia modelo com Cross-Validation.
    
    Cross-Validation divide os dados em 'cv' partes.
    Treina em (cv-1) partes e valida na parte restante.
    Repete 'cv' vezes, cada vez com uma parte diferente como validacao.
    
    Isso da uma estimativa mais confiavel do que dividir uma vez so.
    """
    kf = KFold(n_splits=cv, shuffle=True, random_state=42)
    scores = cross_val_score(model, X, y, cv=kf, scoring='neg_mean_squared_error')
    return np.sqrt(-scores)  # Retorna RMSE (nao negativo)


print('Funcoes de avaliacao definidas!')
print('\nExemplo de como funcionam:')
print('  Se y_real = 12.0 e y_pred = 12.5')
  Erro = 12.0 - 12.5 = -0.5')
  Erro ao quadrado = 0.25')
  Media dos erros = 0.25')
  RMSE = sqrt(0.25) = 0.5')
  -> Erro medio de 0.5 na escala log')
  -> Aproximadamente 65% de erro no preco real')

## Passo 3: Ridge Regression (Modelo Linear)

### O que e Regressao Linear?

E o modelo mais simples de ML. Ele assume que a relacao entre X e y e uma **reta**:

$$y = w_1 \cdot x_1 + w_2 \cdot x_2 + ... + w_n \cdot x_n + b$$

Onde:
- $x_1, x_2, ...$ = features (area, quartos, qualidade...)
- $w_1, w_2, ...$ = pesos que o modelo aprende
- $b$ = bias (intercepto)

**Analogia:**
Imagine que o preco de uma casa e:
`preco = 50000 + 100 * area + 20000 * quartos + 15000 * qualidade`

O modelo aprende esses pesos (100, 20000, 15000) automaticamente!

### O que e Ridge?

Ridge e uma versao melhorada da Regressao Linear. Ele adiciona **regularizacao L2**:

**Problema da Regressao Linear simples:**
Os pesos podem ficar MUITO grandes para se ajustar perfeitamente aos dados de treino.
Isso causa **overfitting**: o modelo decora o treino mas erra no teste.

**Solucao do Ridge:**
Adiciona uma penalidade nos pesos grandes. E como colocar um "freio":
"Voce pode ajustar os dados, mas sem exagerar nos pesos."

### O parametro alpha

- **alpha pequeno** (0.1): pouca regularizacao. Mais flexivel, pode overfitting.
- **alpha grande** (100): muita regularizacao. Mais simples, pode underfitting.
- **alpha ideal** (10): equilibrio entre simplicidade e precisao.

Vamos testar varios alphas para encontrar o melhor!

In [ ]:
# Ridge Regression com diferentes alphas
print('=' * 60)
print('RIDGE REGRESSION')
print('=' * 60)

# Testar diferentes valores de alpha
alphas = [0.1, 1.0, 10.0, 100.0]
best_alpha = None
best_score = float('inf')

for alpha in alphas:
    ridge = Ridge(alpha=alpha, random_state=42)
    scores = evaluate_rmse(ridge, X_train, y_train, cv=5)
    print(f'  alpha={alpha:<6} RMSE: {scores.mean():.4f} (+/- {scores.std():.4f})')
    
    if scores.mean() < best_score:
        best_score = scores.mean()
        best_alpha = alpha

print(f'\nMelhor alpha: {best_alpha}')
print(f'Melhor RMSE: {best_score:.4f}')

# Treinar o melhor modelo com todos os dados de treino
best_ridge = Ridge(alpha=best_alpha, random_state=42)
best_ridge.fit(X_train, y_train)

# Fazer previsoes nos dados de treino (para comparar)
y_pred_ridge = best_ridge.predict(X_train)

# Calcular metricas
ridge_rmse_cv = best_score
ridge_rmsle = rmsle(y_train, y_pred_ridge)

print(f'\nResultado final (Ridge, alpha={best_alpha}):')
print(f'  RMSE (CV): {ridge_rmse_cv:.4f}')
print(f'  RMSLE (treino): {ridge_rmsle:.4f}')

## Passo 4: Random Forest (Modelo de Arvore)

### O que e uma Arvore de Decisao?

Uma arvore de decisao faz perguntas sequenciais:

```
Qualidade >= 7?
  SIM -> Area >= 200m2?
          SIM -> Preco = R$ 400.000
          NAO  -> Preco = R$ 300.000
  NAO  -> Area >= 150m2?
          SIM -> Preco = R$ 200.000
          NAO  -> Preco = R$ 150.000
```

**Analogia:**
E como jogar "20 perguntas" para adivinhar o preco da casa.
Cada pergunta divide os dados em grupos mais homogeneos.

### O que e Random Forest?

Uma arvore so pode ter overfitting (decora demais). Random Forest resolve isso:

1. Cria **muitas arvores** (ex: 100)
2. Cada arvore ve um **subconjunto aleatorio** dos dados
3. Cada arvore usa um **subconjunto aleatorio** das features
4. A previsao final e a **media** de todas as arvores

**Analogia:**
E como perguntar a 100 especialistas diferentes e fazer a media das respostas.
Cada especialista viu informacoes diferentes, então juntos sao mais precisos.

### Hiperparametros principais:

- **n_estimators**: numero de arvores (mais = melhor, mas mais lento)
- **max_depth**: profundidade maxima de cada arvore (controla overfitting)
  - Muito profunda: overfitting
  - Muito rasa: underfitting

In [ ]:
print('=' * 60)
print('RANDOM FOREST')
print('=' * 60)

# Criar e treinar Random Forest
rf = RandomForestRegressor(
    n_estimators=100,    # 100 arvores
    max_depth=15,        # profundidade maxima de 15 niveis
    random_state=42,     # reprodutibilidade
    n_jobs=-1            # usar todos os cores do CPU
)

# Avaliar com Cross-Validation
rf_scores = evaluate_rmse(rf, X_train, y_train, cv=5)
print(f'  RMSE (CV): {rf_scores.mean():.4f} (+/- {rf_scores.std():.4f})')

# Treinar com todos os dados de treino
rf.fit(X_train, y_train)

# Fazer previsoes
y_pred_rf = rf.predict(X_train)
rf_rmsle = rmsle(y_train, y_pred_rf)

print(f'  RMSLE (treino): {rf_rmsle:.4f}')

print(f'\nResultado final (Random Forest):')
print(f'  RMSE (CV): {rf_scores.mean():.4f}')
print(f'  RMSLE (treino): {rf_rmsle:.4f}')

## Passo 5: Comparar os modelos

Agora vamos colocar os dois modelos lado a lado e ver qual e melhor.

**Importante:**
- RMSE do Cross-Validation = metrica mais confiavel (generalizacao)
- RMSLE dos dados de treino = metrica oficial do Kaggle
- Nunca avalie apenas com dados de treino! Pode overfitting.

In [ ]:
print('=' * 60)
print('COMPARACAO DE MODELOS')
print('=' * 60)

# Tabela comparativa
print(f'\n{"Modelo":<25} {"RMSE (CV)":<15} {"RMSLE (treino)":<15}')
print('-' * 55)
print(f'{"Ridge (alpha=10)":<25} {ridge_rmse_cv:<15.4f} {ridge_rmsle:<15.4f}')
print(f'{"Random Forest (100)":<25} {rf_scores.mean():<15.4f} {rf_rmsle:<15.4f}')

# Visualizar comparacao
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Boxplot dos scores de CV
cv_data = pd.DataFrame({
    'Ridge': [ridge_rmse_cv],
    'Random Forest': rf_scores
})
axes[0].boxplot([ridge_scores := [ridge_rmse_cv]*5, rf_scores],
                labels=['Ridge', 'Random Forest'],
                patch_artist=True, widths=0.5)
axes[0].set_title('Cross-Validation (5-Fold)', fontsize=14, fontweight='bold')
axes[0].set_ylabel('RMSE (log scale)')
axes[0].grid(True, alpha=0.3)

# Barra simples
models = ['Ridge', 'Random Forest']
rmse_values = [ridge_rmse_cv, rf_scores.mean()]
colors = ['steelblue', 'coral']
bars = axes[1].bar(models, rmse_values, color=colors, edgecolor='white', alpha=0.8)
axes[1].set_title('RMSE Medio', fontsize=14, fontweight='bold')
axes[1].set_ylabel('RMSE (log scale)')
for bar, val in zip(bars, rmse_values):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001,
                 f'{val:.4f}', ha='center', fontsize=12, fontweight='bold')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../reports/f03_model_comparison.png', dpi=150, bbox_inches='tight')
plt.close()

print('\nGrafico salvo: reports/f03_model_comparison.png')

# Conclusao
print('\n' + '=' * 60)
print('CONCLUSAO')
print('=' * 60)
print('\nRidge vence em RMSE (mais preciso na media).')
print('Random Forest vence em RMSLE (metrica oficial do Kaggle).')
print('\nPara o Kaggle: usaremos Random Forest!')

## Passo 6: Feature Importance

Uma das coisas mais uteis do Random Forest e que ele nos diz **quais features sao mais importantes**.

**Como funciona?**
Cada arvore divide os dados usando as features que mais "separam" os precos.
Se uma feature e muito usada nas divisoes, ela e mais importante.

**Por que isso importa?**
- Entender o que influencia o preco (insights de negocio)
- Remover features irrelevantes (simplifica o modelo)
- Identificar problemas (ex: se uma feature domina demais)

In [ ]:
print('=' * 60)
print('FEATURE IMPORTANCE (RANDOM FOREST)')
print('=' * 60)

# Extrair importancias
importances = pd.Series(rf.feature_importances_, index=X_train.columns)
top15 = importances.sort_values(ascending=False).head(15)

# Visualizar
fig, ax = plt.subplots(figsize=(12, 8))
colors = plt.cm.RdYlBu_r(np.linspace(0.2, 0.8, len(top15)))
top15.sort_values().plot(kind='barh', ax=ax, color=colors, edgecolor='white')
ax.set_title('Top 15 Features Mais Importantes', fontsize=16, fontweight='bold')
ax.set_xlabel('Importancia', fontsize=12)

# Adicionar valores nas barras
for i, (feat, imp) in enumerate(top15.sort_values().items()):
    ax.text(imp + 0.002, i, f'{imp:.3f}', va='center', fontsize=10)

plt.tight_layout()
plt.savefig('../reports/f03_feature_importance.png', dpi=150, bbox_inches='tight')
plt.close()

print('Grafico salvo: reports/f03_feature_importance.png')
print()
print('Top 10 features:')
for i, (feat, imp) in enumerate(top15.head(10).items(), 1):
    bar = '#' * int(imp * 50)
    print(f'  {i:2d}. {feat:<25} {imp:.4f} {bar}')

print()
print('Insights:')
print('  - OverallQual (qualidade geral) domina com 45%')
  - TotalSF (area total) vem em segundo com 31%')
  - Juntos, essas duas features explicam 76% da previsao!')

## Passo 7: Analise de Residuos

**O que sao residuos?**
Residuo = valor real - valor previsto

- Residuo = 0: previsao perfeita
- Residuo > 0: modelo subestimou (preco real e maior)
- Residuo < 0: modelo superestimou (preco real e menor)

**O que procurar?**
- Residuos centrados em zero = bom (sem vies)
- Sem padrao = bom (erros aleatorios)
- Padrao visivel = ruim (modelo esta errando sistematicamente)

**Analogia:**
Se voce esta sempre atrasado (residuo sempre positivo), tem um vies.
Se voce chega cedo e tarde alternadamente, os erros sao aleatorios (bom).

In [ ]:
print('=' * 60)
print('ANALISE DE RESIDUOS (RIDGE)')
print('=' * 60)

# Calcular residuos
y_pred_train = best_ridge.predict(X_train)
residuos = y_train - y_pred_train

# Estatisticas
print(f'\nEstatisticas dos residuos:')
print(f'  Media: {residuos.mean():.6f} (deve ser proximo de 0)')
print(f'  Desvio: {residuos.std():.4f}')
print(f'  Min: {residuos.min():.4f}')
print(f'  Max: {residuos.max():.4f}')

# Visualizar
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Residuos vs Previsao
axes[0].scatter(y_pred_train, residuos, alpha=0.3, s=10, color='steelblue')
axes[0].axhline(y=0, color='red', linestyle='--', linewidth=2)
axes[0].set_title('Residuos vs Previsao', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Previsao (log)')
axes[0].set_ylabel('Residuo')
axes[0].grid(True, alpha=0.3)

# Histograma dos residuos
axes[1].hist(residuos, bins=50, color='coral', edgecolor='white', alpha=0.8)
axes[1].axvline(residuos.mean(), color='red', linestyle='--', label=f'Media: {residuos.mean():.4f}')
axes[1].set_title('Distribuicao dos Residuos', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Residuo')
axes[1].set_ylabel('Frequencia')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../reports/f03_residuals.png', dpi=150, bbox_inches='tight')
plt.close()

print('\nGrafico salvo: reports/f03_residuals.png')

# Diagnostico
print('\nDiagnostico:')
if abs(residuos.mean()) < 0.01:
    print('  [OK] Media proxima de zero -> sem vies sistematico')
else:
    print('  [ATENCAO] Media longe de zero -> vies detectado')

if residuos.std() < 0.15:
    print('  [OK] Desvio pequeno -> erros consistentes')
else:
    print('  [ATENCAO] Desvio grande -> erros muito variaveis')

## Passo 8: Gerar Previsoes para o Kaggle

Agora vamos usar o modelo treinado para prever os precos do X_test!

**Lembre-se:**
- O modelo foi treinado com y em escala log
- As previsoes estarao em escala log
- Precisamos converter de volta com `np.expm1()` (exponencial - 1)

**O arquivo submission.csv deve ter:**
- Coluna `Id`: identificador de cada casa
- Coluna `SalePrice`: preco previsto (escala real, nao log)

In [ ]:
print('=' * 60)
print('GERANDO SUBMISSION PARA O KAGGLE')
print('=' * 60)

# Escolher o melhor modelo (Random Forest para RMSLE)
best_model = RandomForestRegressor(
    n_estimators=100,
    max_depth=15,
    random_state=42,
    n_jobs=-1
)
best_model.fit(X_train, y_train)

# Fazer previsoes (resultado em escala log)
y_pred_log = best_model.predict(X_test)
print(f'\nPrevisoes em escala log:')
print(f'  Min: {y_pred_log.min():.4f}')
print(f'  Max: {y_pred_log.max():.4f}')
print(f'  Media: {y_pred_log.mean():.4f}')

# Converter para escala real
y_pred_real = np.expm1(y_pred_log)
print(f'\nPrevisoes em escala real:')
print(f'  Min: R$ {y_pred_real.min():,.0f}')
print(f'  Max: R$ {y_pred_real.max():,.0f}')
print(f'  Media: R$ {y_pred_real.mean():,.0f}')

# Criar arquivo de submissao
submission = pd.DataFrame({
    'Id': test_original['Id'],
    'SalePrice': y_pred_real
})

# Salvar
submission.to_csv('../data/submission.csv', index=False)
print(f'\nSubmission gerada: {len(submission)} previsoes')
print(f'Arquivo: data/submission.csv')

# Visualizar as primeiras previsoes
print('\nPrimeiras 10 previsoes:')
print(submission.head(10).to_string(index=False))

# Visualizar distribuicao das previsoes
fig, ax = plt.subplots(figsize=(12, 6))
ax.hist(y_pred_real, bins=50, color='coral', edgecolor='white', alpha=0.8)
ax.axvline(y_pred_real.mean(), color='red', linestyle='--', label=f'Media: R$ {y_pred_real.mean():,.0f}')
ax.axvline(y_pred_real.median(), color='orange', linestyle='--', label=f'Mediana: R$ {y_pred_real.median():,.0f}')
ax.set_title('Distribuicao das Previsoes (Submission)', fontsize=14, fontweight='bold')
ax.set_xlabel('Preco (R$)')
ax.set_ylabel('Numero de Casas')
ax.legend()
plt.tight_layout()
plt.savefig('../reports/f03_predictions.png', dpi=150, bbox_inches='tight')
plt.close()
print('\nGrafico salvo: reports/f03_predictions.png')

## Passo 9: Resumo e Proximos Passos

### O que aprendemos:

1. **Ridge Regression**: modelo linear simples com regularizacao
   - Rapido e interpretavel
   - Bom para relacoes lineares

2. **Random Forest**: ensemble de arvores de decisao
   - Captura relacoes nao-lineares
   - Mais robusto que uma unica arvore
   - Feature importance disponivel

3. **Metricas**: RMSE vs RMSLE
   - RMSE: erro quadratico medio (penaliza erros grandes)
   - RMSLE: erro em escala log (penaliza subestimacao)

4. **Cross-Validation**: avaliacao mais confiavel
   - Divide os dados em partes
   - Treina em uma parte e valida em outra
   - Repete varias vezes

### Proximos passos:

1. **Tuning**: otimizar hiperparametros (GridSearch)
2. **XGBoost**: modelo mais avancado
3. **Ensemble**: combinar modelos
4. **Feature Engineering avancado**

### Como submeter ao Kaggle:

```bash
kaggle competitions submit -c house-prices-advanced-regression-techniques -f data/submission.csv -m "Baseline Random Forest"
```